# Phase 17 — Within-Dataset Performance Ceiling (FIXED)

**NetShield**

This notebook is a clean, self-contained control experiment. It fixes the dependency/order problems from the earlier Phase 17 file:

- no undefined `make_models()`
- no undefined `choose_threshold()`
- no undefined `fit_transform_source_target()`
- no undefined `metrics_row()`
- robust CICIDS `Label` detection (whitespace/BOM/case)
- no giant CICIDS dataframe
- no `usecols=["Label"]` failure when the real header has whitespace/BOM
- safe scalar division (no `.replace()` on Python integers)
- all helper functions are defined before experiments run

The experiment establishes the **within-dataset performance ceiling** for the 11-feature and expanded semantic representations using Random Forest, Extra Trees, and HistGradientBoosting. Validation data selects the threshold; the test set is used only once for final evaluation.

In [ ]:
# ============================================================
# 0. Imports, configuration, paths
# ============================================================

from pathlib import Path
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
CICIDS_CHUNK_SIZE = 100_000
CICIDS_SAMPLE_N = 1_000_000

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in candidates:
        if (base / "data" / "raw" / "UNSW-NB15").exists() and (base / "data" / "raw" / "CICIDS2017").exists():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
UNSW_ROOT = PROJECT_ROOT / "data" / "raw" / "UNSW-NB15"
CICIDS_ROOT = PROJECT_ROOT / "data" / "raw" / "CICIDS2017"
OUT = PROJECT_ROOT / "outputs" / "phase17_within_dataset_ceiling_fixed"
TABLES = OUT / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UNSW_ROOT   :", UNSW_ROOT)
print("CICIDS_ROOT :", CICIDS_ROOT)
print("OUTPUT      :", OUT)

## 1. Load UNSW and discover CICIDS files

UNSW's official train/test split is preserved. CICIDS is kept on disk and processed in chunks to avoid the previous memory failures.

In [ ]:
# ============================================================
# 1. Load UNSW + discover CICIDS files
# ============================================================

u_train = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_training-set.csv"
u_test = UNSW_ROOT / "Training and Testing Sets" / "UNSW_NB15_testing-set.csv"

if not u_train.exists():
    matches = list(UNSW_ROOT.rglob("UNSW_NB15_training-set.csv"))
    if not matches:
        raise FileNotFoundError("Could not find UNSW_NB15_training-set.csv")
    u_train = matches[0]

if not u_test.exists():
    matches = list(UNSW_ROOT.rglob("UNSW_NB15_testing-set.csv"))
    if not matches:
        raise FileNotFoundError("Could not find UNSW_NB15_testing-set.csv")
    u_test = matches[0]

unsw_train = pd.read_csv(u_train)
unsw_test = pd.read_csv(u_test)

unsw_train.columns = (unsw_train.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip())
unsw_test.columns = (unsw_test.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip())

if "label" not in unsw_train.columns or "label" not in unsw_test.columns:
    raise KeyError("UNSW train/test must contain a 'label' column.")

unsw_train["label_binary"] = pd.to_numeric(unsw_train["label"], errors="coerce").astype("int8")
unsw_test["label_binary"] = pd.to_numeric(unsw_test["label"], errors="coerce").astype("int8")

cicids_files = sorted(CICIDS_ROOT.rglob("*.csv"))
if not cicids_files:
    raise FileNotFoundError(f"No CICIDS CSV files found under {CICIDS_ROOT}")

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)
print("CICIDS files:", len(cicids_files))
for p in cicids_files:
    print(" -", p.name)

## 2. Robust CICIDS header handling

The earlier failures happened because some CICIDS files do not expose the label/features under the exact string expected by `usecols`. We therefore read only the header first, normalize whitespace/BOM/case, and map requested names to the real column names.

In [ ]:
# ============================================================
# 2. Robust CICIDS schema helpers
# ============================================================

def header_map(path):
    h = pd.read_csv(path, nrows=0, low_memory=True)
    return {
        str(c).replace("\ufeff", "").strip().lower(): c
        for c in h.columns
    }

def actual_col(path, requested):
    mapping = header_map(path)
    key = requested.replace("\ufeff", "").strip().lower()
    if key not in mapping:
        raise KeyError(
            f"Column {requested!r} not found in {path.name}.\n"
            f"Available columns: {list(mapping.values())}"
        )
    return mapping[key]

CICIDS_REQUIRED = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Average Packet Size",
    "Label",
]

for path in cicids_files:
    for name in CICIDS_REQUIRED:
        actual_col(path, name)

print("All CICIDS files passed robust schema verification.")

## 3. Memory-safe CICIDS audit

This is intentionally chunked. It never creates a full 2.8M-row CICIDS dataframe.

In [ ]:
# ============================================================
# 3. Memory-safe CICIDS audit
# ============================================================

audit_rows = []

for path in cicids_files:
    label_col = actual_col(path, "Label")
    rows = 0
    missing_cells = 0
    benign = 0
    attack = 0

    # Read all columns only for the audit, one chunk at a time.
    for chunk in pd.read_csv(path, chunksize=CICIDS_CHUNK_SIZE, low_memory=True):
        chunk.columns = (chunk.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip())
        rows += len(chunk)
        missing_cells += int(chunk.isna().sum().sum())
        labels = chunk[label_col].astype(str).str.strip().str.upper()
        benign += int((labels == "BENIGN").sum())
        attack += int((labels != "BENIGN").sum())
        del labels, chunk
        gc.collect()

    audit_rows.append({
        "file": path.name,
        "rows": rows,
        "missing_cells": missing_cells,
        "benign": benign,
        "attack": attack,
        "attack_rate": attack / rows if rows else np.nan,
    })
    print(f"Audited: {path.name} | rows={rows:,}")

cicids_audit = pd.DataFrame(audit_rows)
TOTAL_CICIDS_ROWS = int(cicids_audit["rows"].sum())
TOTAL_CICIDS_BENIGN = int(cicids_audit["benign"].sum())
TOTAL_CICIDS_ATTACK = int(cicids_audit["attack"].sum())

print("\nCICIDS total rows :", TOTAL_CICIDS_ROWS)
print("CICIDS benign     :", TOTAL_CICIDS_BENIGN)
print("CICIDS attack     :", TOTAL_CICIDS_ATTACK)
print("CICIDS attack rate:", round(TOTAL_CICIDS_ATTACK / TOTAL_CICIDS_ROWS, 6))

display(cicids_audit)
cicids_audit.to_csv(TABLES / "01_cicids_audit.csv", index=False)

## 4. Memory-safe stratified CICIDS sample

We keep at most 1,000,000 rows, preserving the global benign/attack prior. Importantly, the sampler reads only the 9 semantic source features plus `Label`; it never loads all CICIDS columns.

In [ ]:
# ============================================================
# 4. Global class counts + memory-safe sampling
# ============================================================

global_counts = {0: 0, 1: 0}

for path in cicids_files:
    label_col = actual_col(path, "Label")
    for chunk in pd.read_csv(path, usecols=[label_col], chunksize=CICIDS_CHUNK_SIZE, low_memory=True):
        labels = chunk[label_col].astype(str).str.strip().str.upper()
        global_counts[0] += int((labels == "BENIGN").sum())
        global_counts[1] += int((labels != "BENIGN").sum())
        del labels, chunk
    gc.collect()

TOTAL_FROM_LABEL_SCAN = sum(global_counts.values())
if TOTAL_FROM_LABEL_SCAN != TOTAL_CICIDS_ROWS:
    raise ValueError(
        f"Audit/label scan mismatch: audit={TOTAL_CICIDS_ROWS}, label_scan={TOTAL_FROM_LABEL_SCAN}"
    )

sample_n = min(CICIDS_SAMPLE_N, TOTAL_CICIDS_ROWS)
target_quota = {
    0: int(round(sample_n * global_counts[0] / TOTAL_CICIDS_ROWS)),
    1: 0,
}
target_quota[1] = sample_n - target_quota[0]

print("Global class counts:", global_counts)
print("Sample size:", sample_n)
print("Target quotas:", target_quota)

FEATURE_NAMES = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Average Packet Size",
]

# We allocate quotas to files proportional to their class counts, then sample
# from each chunk. This avoids the previous `Label`/usecols errors.
file_counts = cicids_audit.set_index("file").to_dict("index")
sample_parts = []

for path in cicids_files:
    meta = file_counts[path.name]
    q0 = int(round(target_quota[0] * meta["benign"] / max(global_counts[0], 1)))
    q1 = int(round(target_quota[1] * meta["attack"] / max(global_counts[1], 1)))

    real_cols = [actual_col(path, c) for c in FEATURE_NAMES + ["Label"]]
    rename_map = {real: requested for real, requested in zip(real_cols, FEATURE_NAMES + ["Label"])}

    file_parts = []
    for chunk in pd.read_csv(path, usecols=real_cols, chunksize=CICIDS_CHUNK_SIZE, low_memory=True):
        chunk = chunk.rename(columns=rename_map)
        labels = chunk["Label"].astype(str).str.strip().str.upper()
        chunk["label_binary"] = (labels != "BENIGN").astype("int8")

        for cls, quota, class_total in [(0, q0, meta["benign"]), (1, q1, meta["attack"])]:
            if quota <= 0 or class_total <= 0:
                continue
            sub = chunk.loc[chunk["label_binary"] == cls]
            if len(sub) == 0:
                continue
            take = min(len(sub), max(0, int(round(quota * len(sub) / class_total))))
            if take > 0:
                file_parts.append(sub.sample(n=take, random_state=RANDOM_STATE))

        del labels, chunk
        gc.collect()

    if file_parts:
        sample_parts.append(pd.concat(file_parts, ignore_index=True))
    del file_parts
    gc.collect()
    print(f"Sampled from: {path.name}")

if not sample_parts:
    raise RuntimeError("No CICIDS rows were sampled.")

cicids_sample = pd.concat(sample_parts, ignore_index=True)
del sample_parts
gc.collect()

# Enforce the exact requested sample size as closely as possible while preserving class prior.
selected = []
for cls, quota in target_quota.items():
    sub = cicids_sample.loc[cicids_sample["label_binary"] == cls]
    if len(sub) < quota:
        raise ValueError(f"Sampler produced only {len(sub)} rows for class {cls}, need {quota}.")
    selected.append(sub.sample(n=quota, random_state=RANDOM_STATE))

cicids_sample = pd.concat(selected, ignore_index=True)
del selected
gc.collect()

print("Final CICIDS sample:", cicids_sample.shape)
display(cicids_sample["label_binary"].value_counts().rename("count").to_frame())

## 5. Verified 11-feature semantic representation

The mapping is deliberately restricted to quantities with clear semantic correspondence between UNSW-NB15 and CICIDS2017.

In [ ]:
# ============================================================
# 5. Common semantic representation
# ============================================================

COMMON_11 = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
    "avg_packet_size",
]

def safe_divide(a, b):
    """Series-safe division; also works when b is a scalar."""
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    if np.isscalar(b):
        if b == 0:
            return a * np.nan
        return a / b
    b = b.mask(b == 0, np.nan)
    return a / b

def build_unsw_common(df):
    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce")
    x["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce")
    x["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce")
    x["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce")
    x["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce")
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = safe_divide(x["total_packets"], x["flow_duration_sec"])
    x["fwd_packet_rate"] = safe_divide(x["fwd_packets"], x["flow_duration_sec"])
    x["bwd_packet_rate"] = safe_divide(x["bwd_packets"], x["flow_duration_sec"])
    x["avg_packet_size"] = safe_divide(x["total_bytes"], x["total_packets"])
    return x.replace([np.inf, -np.inf], np.nan).astype("float32")

def build_cicids_common(df):
    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = safe_divide(df["Flow Duration"], 1_000_000)
    x["fwd_packets"] = pd.to_numeric(df["Total Fwd Packets"], errors="coerce")
    x["bwd_packets"] = pd.to_numeric(df["Total Backward Packets"], errors="coerce")
    x["fwd_bytes"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce")
    x["bwd_bytes"] = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce")
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = pd.to_numeric(df["Flow Packets/s"], errors="coerce")
    x["fwd_packet_rate"] = pd.to_numeric(df["Fwd Packets/s"], errors="coerce")
    x["bwd_packet_rate"] = pd.to_numeric(df["Bwd Packets/s"], errors="coerce")
    x["avg_packet_size"] = pd.to_numeric(df["Average Packet Size"], errors="coerce")
    return x.replace([np.inf, -np.inf], np.nan).astype("float32")

X_unsw_train_11 = build_unsw_common(unsw_train)
X_unsw_test_11 = build_unsw_common(unsw_test)
X_cicids_11 = build_cicids_common(cicids_sample)

y_unsw_train = unsw_train["label_binary"].to_numpy(dtype="int8")
y_unsw_test = unsw_test["label_binary"].to_numpy(dtype="int8")
y_cicids = cicids_sample["label_binary"].to_numpy(dtype="int8")

print("UNSW train 11:", X_unsw_train_11.shape)
print("UNSW test  11:", X_unsw_test_11.shape)
print("CICIDS 11    :", X_cicids_11.shape)

## 6. Expanded semantic representation

All extra variables are deterministic functions of the verified 11 quantities. No target labels are used.

In [ ]:
# ============================================================
# 6. Expanded semantic representation
# ============================================================

def build_expanded_semantic(x11):
    x = x11.copy()
    eps = 1e-9

    x["fwd_packet_fraction"] = x["fwd_packets"] / (x["total_packets"] + eps)
    x["bwd_packet_fraction"] = x["bwd_packets"] / (x["total_packets"] + eps)
    x["fwd_byte_fraction"] = x["fwd_bytes"] / (x["total_bytes"] + eps)
    x["bwd_byte_fraction"] = x["bwd_bytes"] / (x["total_bytes"] + eps)
    x["fwd_bwd_packet_ratio"] = x["fwd_packets"] / (x["bwd_packets"] + eps)
    x["fwd_bwd_byte_ratio"] = x["fwd_bytes"] / (x["bwd_bytes"] + eps)
    x["fwd_bytes_per_packet"] = x["fwd_bytes"] / (x["fwd_packets"] + eps)
    x["bwd_bytes_per_packet"] = x["bwd_bytes"] / (x["bwd_packets"] + eps)
    x["bytes_per_packet"] = x["total_bytes"] / (x["total_packets"] + eps)

    duration = x["flow_duration_sec"].mask(x["flow_duration_sec"] == 0, np.nan)
    x["byte_rate"] = x["total_bytes"] / duration
    x["fwd_byte_rate"] = x["fwd_bytes"] / duration
    x["bwd_byte_rate"] = x["bwd_bytes"] / duration

    log_cols = [
        "flow_duration_sec", "fwd_packets", "bwd_packets", "fwd_bytes",
        "bwd_bytes", "total_packets", "total_bytes", "packet_rate",
        "fwd_packet_rate", "bwd_packet_rate", "avg_packet_size",
        "byte_rate", "fwd_byte_rate", "bwd_byte_rate"
    ]
    for col in log_cols:
        x[f"log1p_{col}"] = np.log1p(x[col].clip(lower=0))

    return x.replace([np.inf, -np.inf], np.nan).astype("float32")

X_unsw_train_exp = build_expanded_semantic(X_unsw_train_11)
X_unsw_test_exp = build_expanded_semantic(X_unsw_test_11)
X_cicids_exp = build_expanded_semantic(X_cicids_11)

print("11 features      :", X_unsw_train_11.shape[1])
print("Expanded features:", X_unsw_train_exp.shape[1])

## 7. Within-dataset splits

UNSW preserves the official test set. Its official training set is split 80/20 for fitting and validation. CICIDS uses a stratified 70/15/15 split of the sampled pool.

In [ ]:
# ============================================================
# 7. Leakage-safe train/validation/test splits
# ============================================================

unsw_fit_idx, unsw_val_idx = train_test_split(
    np.arange(len(y_unsw_train)),
    test_size=0.20,
    stratify=y_unsw_train,
    random_state=RANDOM_STATE,
)

cic_train_idx, cic_temp_idx = train_test_split(
    np.arange(len(y_cicids)),
    test_size=0.30,
    stratify=y_cicids,
    random_state=RANDOM_STATE,
)

cic_val_idx, cic_test_idx = train_test_split(
    cic_temp_idx,
    test_size=0.50,
    stratify=y_cicids[cic_temp_idx],
    random_state=RANDOM_STATE,
)

print("UNSW fit/validation:", len(unsw_fit_idx), len(unsw_val_idx))
print("CICIDS train/val/test:", len(cic_train_idx), len(cic_val_idx), len(cic_test_idx))

## 8. All experiment helper functions

**This cell is intentionally before every model experiment.** This prevents the `NameError` problems from the earlier notebook.

In [ ]:
# ============================================================
# 8. Preprocessing, models, threshold, metrics
# ============================================================

def fit_transform_source_target(X_fit, X_val, X_test):
    """Fit median imputation on fit data only; transform val/test with it."""
    imputer = SimpleImputer(strategy="median")
    Xf = imputer.fit_transform(X_fit).astype("float32")
    Xv = imputer.transform(X_val).astype("float32")
    Xt = imputer.transform(X_test).astype("float32")
    return Xf, Xv, Xt, imputer

def make_models():
    """Fresh tree-based models for every representation/experiment."""
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=250,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2,
        ),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=250,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            max_features="sqrt",
            min_samples_leaf=2,
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        ),
    }

def choose_threshold(y_true, probabilities):
    """Choose the threshold maximizing validation F1 only."""
    thresholds = np.arange(0.05, 0.96, 0.01)
    best_threshold = 0.50
    best_f1 = -1.0
    for threshold in thresholds:
        pred = (probabilities >= threshold).astype("int8")
        score = f1_score(y_true, pred, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = threshold
    return float(best_threshold)

def metrics_row(y_true, probabilities, threshold, direction, representation, model_name):
    pred = (probabilities >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "direction": direction,
        "representation": representation,
        "model": model_name,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "pr_auc": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "predicted_attack_rate": float(pred.mean()),
    }

print("All helper functions loaded successfully.")
print("Models:", list(make_models().keys()))
print("Threshold selector: ready")

## 9. Experiment A — UNSW within-dataset ceiling

In [ ]:
# ============================================================
# 9. Experiment A — UNSW within-dataset ceiling
# ============================================================

unsw_results = []
reps_unsw = {
    "11-semantic": (X_unsw_train_11, X_unsw_test_11),
    "expanded-semantic": (X_unsw_train_exp, X_unsw_test_exp),
}

yfit = y_unsw_train[unsw_fit_idx]
yval = y_unsw_train[unsw_val_idx]

for rep, (X_all, X_test) in reps_unsw.items():
    Xf, Xv, Xt, _ = fit_transform_source_target(
        X_all.iloc[unsw_fit_idx],
        X_all.iloc[unsw_val_idx],
        X_test,
    )

    for model_name, model in make_models().items():
        t0 = time.perf_counter()
        model.fit(Xf, yfit)
        train_seconds = time.perf_counter() - t0

        pval = model.predict_proba(Xv)[:, 1]
        threshold = choose_threshold(yval, pval)
        ptest = model.predict_proba(Xt)[:, 1]

        row = metrics_row(
            y_unsw_test, ptest, threshold,
            "UNSW_within", rep, model_name
        )
        row["train_seconds"] = train_seconds
        unsw_results.append(row)

res_unsw_within = pd.DataFrame(unsw_results)
display(res_unsw_within.sort_values("f1", ascending=False))

## 10. Experiment B — CICIDS within-dataset ceiling

In [ ]:
# ============================================================
# 10. Experiment B — CICIDS within-dataset ceiling
# ============================================================

cic_results = []
reps_cic = {
    "11-semantic": X_cicids_11,
    "expanded-semantic": X_cicids_exp,
}

yfit = y_cicids[cic_train_idx]
yval = y_cicids[cic_val_idx]
ytest = y_cicids[cic_test_idx]

for rep, X in reps_cic.items():
    Xf, Xv, Xt, _ = fit_transform_source_target(
        X.iloc[cic_train_idx],
        X.iloc[cic_val_idx],
        X.iloc[cic_test_idx],
    )

    for model_name, model in make_models().items():
        t0 = time.perf_counter()
        model.fit(Xf, yfit)
        train_seconds = time.perf_counter() - t0

        pval = model.predict_proba(Xv)[:, 1]
        threshold = choose_threshold(yval, pval)
        ptest = model.predict_proba(Xt)[:, 1]

        row = metrics_row(
            ytest, ptest, threshold,
            "CICIDS_within", rep, model_name
        )
        row["train_seconds"] = train_seconds
        cic_results.append(row)

res_cic_within = pd.DataFrame(cic_results)
display(res_cic_within.sort_values("f1", ascending=False))

## 11. Save and summarize the within-dataset ceiling

In [ ]:
# ============================================================
# 11. Final results + saved outputs
# ============================================================

within_results = pd.concat(
    [res_unsw_within, res_cic_within],
    ignore_index=True,
)

within_results = within_results.sort_values(
    ["direction", "f1"], ascending=[True, False]
).reset_index(drop=True)

within_results.to_csv(
    TABLES / "03_within_dataset_ceiling_results.csv",
    index=False,
)

best = (
    within_results
    .sort_values("f1", ascending=False)
    .groupby("direction", as_index=False)
    .first()
)

best.to_csv(
    TABLES / "04_best_within_dataset_models.csv",
    index=False,
)

config = {
    "random_state": RANDOM_STATE,
    "cicids_sample_n": int(CICIDS_SAMPLE_N),
    "cicids_actual_sample_n": int(len(cicids_sample)),
    "representations": {"compact": 11, "expanded": int(X_unsw_train_exp.shape[1])},
    "models": list(make_models().keys()),
    "threshold_selection": "validation F1; test labels never used for threshold selection",
    "unsw_split": "official train/test; 20% of official train used as validation",
    "cicids_split": "70/15/15 stratified split of sampled pool",
}

with open(OUT / "05_experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("\nBEST WITHIN-DATASET MODELS")
display(best[[
    "direction", "representation", "model", "f1", "macro_f1",
    "precision", "recall", "pr_auc", "roc_auc", "fpr",
    "accuracy", "threshold", "train_seconds"
]])

print("\nSaved outputs to:", TABLES)

## 12. Interpretation checkpoint

Use this phase as a control. If within-dataset F1/PR-AUC is strong while cross-dataset performance remains much lower, the main bottleneck is dataset shift and/or target decision calibration rather than a fundamentally incapable classifier. If expanded semantic features clearly improve both within-dataset ceilings, they are justified for the next cross-dataset intervention. If they do not, keep the compact 11-feature representation as the simpler baseline.